<a href="https://colab.research.google.com/github/Donvicton/Lista-2-Intelig-ncia-Artificial/blob/main/Quest%C3%A3o%203/Agente_de_Viagem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Configurações Iniciais e Imports
Primeiro, vamos instalar e importar as bibliotecas necessárias.

OBS: Deve-se criar Secrets das API da LLM e do OpenWeather.

In [ ]:
import google.generativeai as genai
import requests
from google.colab import userdata
import json

try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    OPENWEATHER_API_KEY = userdata.get('OPENWEATHER_API_KEY')

    genai.configure(api_key=GOOGLE_API_KEY)

    # Usando o nome estável do modelo para evitar erro 404
    model = genai.GenerativeModel('gemini-2.5-flash')

    print("✅ APIs configuradas com sucesso com o modelo Gemini 2.5 Flash.")
except Exception as e:
    print(f"❌ Erro ao configurar APIs: {e}")

✅ APIs configuradas com sucesso com o modelo Gemini 2.5 Flash.


### 2. Função de Clima
Esta função buscará o clima atual de uma cidade específica.

In [ ]:
def get_weather(city):
    """Busca o clima atual usando a API OpenWeatherMap."""
    base_url = "http://api.openweathermap.org/data/2.5/weather"
    params = {
        'q': city,
        'appid': OPENWEATHER_API_KEY,
        'units': 'metric',
        'lang': 'pt_br'
    }
    try:
        response = requests.get(base_url, params=params)
        data = response.json()
        if response.status_code == 200:
            clima = data['weather'][0]['description']
            temp = data['main']['temp']
            return f"O clima em {city} está {clima} com temperatura de {temp}°C."
        else:
            return "Não foi possível obter informações climáticas no momento."
    except Exception as e:
        return f"Erro ao conectar com a API de clima: {e}"

### 3. O Agente de Viagens
Agora, definimos a lógica que une o clima ao LLM para dar a dica.

OBS: Graças ao historico de conversa nosso modelo é classificado como Agente baseado em Modelo tendo a LLM processando toda a parte lógica e o mundo atual.

In [ ]:
# Estado Interno: Lista para armazenar o histórico da conversa
historico_conversa = []

In [ ]:
def travel_agent(user_input, city):
    global historico_conversa

    # Construindo o contexto histórico para o prompt
    contexto_historico_str = "\n".join([f"Usuário: {h['pergunta']}\nAgente: {h['resposta']}" for h in historico_conversa])

    weather_context = get_weather(city)

    prompt = f"""
    Você é um agente de viagens especialista e simpático.

    Contexto do Clima Atual em {city}: {weather_context}

    Histórico da Conversa:
    {contexto_historico_str}

    Nova Pergunta do Usuário: {user_input}

    Com base no clima, no histórico acima e na nova pergunta, forneça dicas práticas.
    """

    try:
        response = model.generate_content(prompt)
        resposta_texto = response.text

        # Atualizando o estado interno (Memória)
        historico_conversa.append({"pergunta": user_input, "resposta": resposta_texto})

        return resposta_texto
    except Exception as e:
        return f"Erro ao gerar resposta: {e}"

# --- Interface de Execução ---
destino = "paris" # @param {type:"string"}
pergunta = "o que voce recomenda para ir ao Jardin du Luxembourg" # @param {type:"string"}

if OPENWEATHER_API_KEY and GOOGLE_API_KEY:
    print(f"\n--- Interação com o Agente (Cidade: {destino}) ---\n")
    resultado = travel_agent(pergunta, destino)
    print(resultado)
else:
    print("Verifique suas chaves de API nos Secrets.")


--- Interação com o Agente (Cidade: paris) ---

Olá novamente! É uma excelente ideia considerar o Jardin du Luxembourg – é um dos meus lugares favoritos em Paris pela sua beleza e atmosfera!

No entanto, com os **37.46°C** que estamos enfrentando, mesmo com o céu nublado, é crucial adaptar a sua visita para que seja agradável e segura. O Jardin du Luxembourg, apesar de ter áreas com sombra, também possui muitas áreas abertas que se tornam verdadeiros "fornos" sob este calor intenso.

Aqui estão minhas recomendações específicas para aproveitar o Jardin du Luxembourg com o calor de hoje:

1.  **Escolha o Melhor Horário – E Seja Estratégico!**
    *   **Início da Manhã (Ideal):** Se você puder ir logo na abertura, por volta das **7h ou 8h da manhã**, você pegará o jardim mais vazio e com as temperaturas ainda mais amenas (embora ainda quentes, estarão no ponto mais baixo do dia). É o melhor momento para uma caminhada rápida e para sentir a energia do lugar antes do calor apertar.
    *  